# Метрики эффективности моделей машинного обучения

## Цель работы

Научиться измерять эффективность моделей машинного обучения с помощью метрик, выбирать метрики исходя из задачи, разбивать датасет на обучающую и тестовую подвыборки.

## 1. Загрузка данных

Загрузим датасет о вероятности развития сердечного приступа:

In [6]:
import pandas as pd
import numpy as np

data = pd.read_csv('https://raw.githubusercontent.com/koroteevmv/ML_course/main/ML4.1%20metrics/data/heart.csv')
data.head()

,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,oldpeak,slp,caa,thall,output
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


Посмотрим на последние строки:

In [7]:
data.tail()

,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,oldpeak,slp,caa,thall,output
298,57,0,0,140,241,0,1,123,1,0.2,1,0,3,0
299,45,1,3,110,264,0,1,132,0,1.2,1,0,3,0
300,68,1,0,144,193,1,1,141,0,3.4,1,2,3,0
301,57,1,0,130,131,0,1,115,1,1.2,1,1,3,0
302,57,0,1,130,236,0,0,174,0,0.0,1,1,2,0


## 2. Обучение модели без разделения выборки

Выделим целевую переменную и обучим модель логистической регрессии на всех данных:

In [8]:
y = data["output"]
x = data.drop("output", axis=1)

from sklearn.linear_model import LogisticRegression

logistic = LogisticRegression(max_iter=1000).fit(x, y)
logistic.score(x, y)

c:\Users\w1nore\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.8481848184818482

Оценка ~85%, но она завышена, так как модель оценивается на тех же данных, на которых обучалась.

## 3. Разделение выборки вручную (первые 200 строк)

In [9]:
x_train, y_train = x[:200], y[:200]
x_test, y_test = x[200:], y[200:]

print('Train shape:', x_train.shape, y_train.shape)
print('Test shape:', x_test.shape, y_test.shape)

Train shape: (200, 13) (200,)
Test shape: (103, 13) (103,)


Обучим и оценим модель:

In [10]:
logistic_test = LogisticRegression(max_iter=1000).fit(x_train, y_train)
print('Train accuracy:', logistic_test.score(x_train, y_train))
print('Test accuracy:', logistic_test.score(x_test, y_test))

Train accuracy: 0.905
Test accuracy: 0.5242718446601942


## 4. Разделение выборки 80/20 вручную

In [11]:
N = int(x.shape[0] * 0.8)
x_train, y_train, x_test, y_test = x[:N], y[:N], x[N:], y[N:]
print('Shapes:', x_train.shape, y_train.shape, x_test.shape, y_test.shape)

Shapes: (242, 13) (242,) (61, 13) (61,)


Обучим и оценим модель:

In [12]:
logistic_test = LogisticRegression(max_iter=1000).fit(x_train, y_train)
print('Train accuracy:', logistic_test.score(x_train, y_train))
print('Test accuracy:', logistic_test.score(x_test, y_test))

Train accuracy: 0.8925619834710744
Test accuracy: 0.6229508196721312


## 5. Случайное разделение выборки с помощью маски

In [13]:
mask = np.array([True] * N + [False] * (y.shape[0] - N))

from numpy.random import shuffle
shuffle(mask)

x_train, y_train, x_test, y_test = x[mask], y[mask], x[~mask], y[~mask]
print('Shapes:', x_train.shape, y_train.shape, x_test.shape, y_test.shape)

Shapes: (242, 13) (242,) (61, 13) (61,)


Обучим и оценим модель после случайного разделения:

In [14]:
logistic_test = LogisticRegression(max_iter=1000).fit(x_train, y_train)
print('Train accuracy:', logistic_test.score(x_train, y_train))
print('Test accuracy:', logistic_test.score(x_test, y_test))

Train accuracy: 0.8471074380165289
Test accuracy: 0.8360655737704918


## 6. Разделение с помощью train_test_split из sklearn

In [15]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.8, random_state=42)
print('Shapes:', x_train.shape, y_train.shape, x_test.shape, y_test.shape)

Shapes: (242, 13) (242,) (61, 13) (61,)


## 7. Построение метрик качества классификации

Импортируем метрики:

In [16]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

Обучим модель на случайном разбиении:

In [17]:
logistic_test = LogisticRegression(max_iter=1000).fit(x_train, y_train)
y_test_pred = logistic_test.predict(x_test)
y_train_pred = logistic_test.predict(x_train)

c:\Users\w1nore\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Матрица ошибок (обучающая выборка):

In [18]:
confusion_matrix(y_train, y_train_pred)

array([[ 87,  22],
       [ 11, 122]])

### Матрица ошибок (тестовая выборка):

In [19]:
confusion_matrix(y_test, y_test_pred)

array([[25,  4],
       [ 3, 29]])

### Отчёт о классификации:

In [20]:
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.89      0.86      0.88        29
           1       0.88      0.91      0.89        32

    accuracy                           0.89        61
   macro avg       0.89      0.88      0.88        61
weighted avg       0.89      0.89      0.89        61



### Подсчёт всех метрик

In [21]:
metrics = pd.DataFrame({
    "Train": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred),
        recall_score(y_train, y_train_pred),
        f1_score(y_train, y_train_pred),
    ],
    "Test": [
        accuracy_score(y_test, y_test_pred),
        precision_score(y_test, y_test_pred),
        recall_score(y_test, y_test_pred),
        f1_score(y_test, y_test_pred),
    ],
}, index=["Accuracy", "Precision", "Recall", "F1"])

metrics

,Train,Test
Accuracy,0.863636,0.885246
Precision,0.847222,0.878788
Recall,0.917293,0.906250
F1,0.880866,0.892308
